In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import einops as eo
import os
import rasterio as rio
from rasterio.features import rasterize
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

from c2s.models.chicken import ChickenConfig, ChickenOutput, Chicken

In [2]:
# Data pipeline parameters

# Data directory - should contain a subdirectory for each site
data_dir = 'data'

# File to put CSV data into
csv_file = 'dataset_file.csv'

# Classes can be commented out here if we wish to exclude them
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}

In [3]:
sites = os.listdir('data')
#sites = ['CS3A'] # override with a single site for testing purposes

In [4]:
def load_site_data(site: str):
    """Retrieves all the rgb / canopy height / label data for a drone site"""
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

    # load RGB raster and metadata
    with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        rgb_raster = f.read((1,2,3))
    
    # load chm file
    with rio.open(chm_file) as f:
        chm_raster = f.read(
            1,
            out_shape=(
                height,
                width
            ),
            resampling=Resampling.nearest
        ).astype(int)
    
    # labels are stored as shapefiles, so we will need to rasterize it
    label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class'])
    label_raster = rasterize(
        ((r['geometry'], r['class']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    return rgb_raster, chm_raster, label_raster

In [5]:
def extract_pixel_features(rgb, chm, labels):
    # Get a list of indices where we have labelled pixels
    # Ex: [ [348, 1451], [581, 2099], ... ]
    indices = np.argwhere(np.isin(labels, list(class_map.values())))
    
    rgb_features = rgb[:,indices[:,0],indices[:,1]].T
    rgb_df = pd.DataFrame(rgb_features, columns=['R', 'G', 'B'])
    
    chm_features = chm[indices[:,0],indices[:,1]].reshape(-1, 1)
    chm_df = pd.DataFrame(chm_features, columns=['chm'])
    
    labels_array = labels[indices[:,0],indices[:,1]].reshape(-1, 1)
    labels_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    df = pd.concat([rgb_df, chm_df, labels_df], axis=1)
    return df, indices

In [6]:
def get_chicken_feature_map(rgb_raster):
    """ Use sliding-window DinoV2 to extract visual features
    rgb_raster: torch tensor with shape (B, C, H, W)
    returns:    torch tensor with shape (B, num_features, H, W)
    """
    rgb_torch_raster = torch.from_numpy(rgb_raster.astype(np.float32)).unsqueeze(0)
    b, c, h, w = rgb_torch_raster.shape
    new_h = int(h/14)*14
    new_w = int(w/14)*14
    rgb_cropped = rgb_torch_raster[:,:,:new_h,:new_w]
    chicken_config = ChickenConfig(
        vit_size="small",
        slider_buffer_size=1
    )
    chicken = Chicken(chicken_config)
    op = chicken._chicken_slider_infer(rgb_cropped).detach().numpy()
    return op

In [7]:
def extract_chicken_pixel_features(chicken_feature_raster, indices):
    # Find the relevant feature vector for labeled pixels. Since texture featuremap is downsampled,
    # we need to modify the indices we use.
    chicken_indices = (indices/14).astype(int)
    chicken_pixel_features = chicken_feature_raster[0][:,chicken_indices[:,0],chicken_indices[:,1]].T
    chicken_df = pd.DataFrame(chicken_pixel_features, columns=[f'chicken_{i}' for i in range(chicken_pixel_features.shape[1])])
    return chicken_df

In [9]:
#all_features = pd.read_csv(csv_file)
#print(sites[11:])

['CS-46B', 'F3-20A', 'F3-8C', 'CS-103A', 'CG1-8A', 'ZF46-15A', 'F3-19B', 'CG1-8B', 'ZF20-11A', 'ZF46-37A', 'CS-46A', 'CS3B', 'F3-19C']


In [10]:
all_features = None
for site in sites[11:]:
    print(f'Processing {site}...')
    rgb, chm, labels = load_site_data(site)
    df1, indices = extract_pixel_features(rgb, chm, labels)
    df1['site'] = site
    
    # texture features
    chicken_feature_map = get_chicken_feature_map(rgb)
    df2 = extract_chicken_pixel_features(chicken_feature_map, indices)

    df = pd.concat([df1,df2], axis=1)

    if all_features is None:
        all_features = df
    else:
        all_features = pd.concat([all_features, df], axis=0)

    all_features.to_csv(csv_file, index=False)
    

Processing CS-46B...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main
xFormers not available
xFormers not available


Processing F3-20A...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


Processing F3-8C...


...

Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


Processing CS3B...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


Processing F3-19C...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


In [11]:
all_features['chm'].value_counts()

chm
 1       18391
 2        5175
 3        4382
-9999      182
Name: count, dtype: int64

In [12]:
# Some additional cleanup
df = all_features
# Remove -9999 values from chm
df['chm'][df['chm'] == -9999] = -1

df['chm'].value_counts()

/tmp/ipykernel_411443/3560184362.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['chm'][df['chm'] == -9999] = -1


chm
 1    18391
 2     5175
 3     4382
-1      182
Name: count, dtype: int64

In [13]:
# Normalize colors
for c in ['R', 'G', 'B']:
    df[c] = df[c].astype(float)/255.0

In [16]:
df

,R,G,B,chm,veg_class,site,chicken_0,chicken_1,chicken_2,chicken_3,...,chicken_374,chicken_375,chicken_376,chicken_377,chicken_378,chicken_379,chicken_380,chicken_381,chicken_382,chicken_383
0,0.874510,0.894118,0.960784,3,13,F3-20C,-0.336462,-0.577480,-0.300406,-0.397835,...,-1.418524,-2.522666,0.922427,-0.972212,0.709010,0.154232,4.143773,2.115928,1.790636,0.175453
1,0.745098,0.776471,0.850980,3,13,F3-20C,-0.336462,-0.577480,-0.300406,-0.397835,...,-1.418524,-2.522666,0.922427,-0.972212,0.709010,0.154232,4.143773,2.115928,1.790636,0.175453
2,0.643137,0.631373,0.713725,3,13,F3-20C,-0.336462,-0.577480,-0.300406,-0.397835,...,-1.418524,-2.522666,0.922427,-0.972212,0.709010,0.154232,4.143773,2.115928,1.790636,0.175453
3,0.811765,0.843137,0.913725,3,13,F3-20C,-0.336462,-0.577480,-0.300406,-0.397835,...,-1.418524,-2.522666,0.922427,-0.972212,0.709010,0.154232,4.143773,2.115928,1.790636,0.175453
4,0.670588,0.701961,0.627451,3,6,F3-20C,-0.210313,-0.434953,0.897604,-0.080942,...,-2.619998,2.019035,1.044443,-2.725528,0.098547,0.100745,2.165668,5.715677,1.534509,0.228802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,0.501961,0.580392,0.439216,1,12,F3-19C,-0.470676,-1.334802,0.251117,0.431643,...,-0.193250,-1.401840,-1.816668,-1.646192,0.397191,-0.480272,2.251428,-2.801445,-2.079341,-0.280176
1171,0.443137,0.415686,0.243137,1,9,F3-19C,3.374305,-0.774874,-1.242118,-1.066557,...,2.770752,-1.204411,0.110569,-0.733905,-3.153169,0.136099,-1.322096,-2.465709,-2.193559,-0.327994
1172,0.447059,0.400000,0.223529,1,9,F3-19C,3.374305,-0.774874,-1.242118,-1.066557,...,2.770752,-1.204411,0.110569,-0.733905,-3.153169,0.136099,-1.322096,-2.465709,-2.193559,-0.327994
1173,0.482353,0.447059,0.266667,1,9,F3-19C,3.374305,-0.774874,-1.242118,-1.066557,...,2.770752,-1.204411,0.110569,-0.733905,-3.153169,0.136099,-1.322096,-2.465709,-2.193559,-0.327994


In [15]:
df.to_csv(csv_file, index=False)